In [ ]:
from typing import Literal

from pydantic import BaseModel
from pydantic_ai import Agent

In [ ]:
PosHint = Literal["noun", "verb", "adjective", "adverb", "other"]

In [ ]:
class SplitEntry(BaseModel):
    hint_form: str
    translations: list[str]
    pos_hint: PosHint


class SplitDecision(BaseModel):
    should_split: bool
    entries: list[SplitEntry]

In [ ]:
system_prompt = """You are a vocabulary analyst for language learning.

Given a word in the target language and its translations, decide if it represents 
multiple distinct meanings that require separate vocabulary entries.

Split when: the same word form covers genuinely different meanings with different 
parts of speech (e.g. a noun AND a verb with different lemmas).

Do NOT split when: translations are just synonyms or register variants of the same meaning.

For each entry after splitting, provide:
- hint_form: the word as it should appear (e.g. 'el cuento' for noun, 'contar' for verb)
- translations: the subset of translations relevant to this entry
- pos_hint: the part of speech

If should_split is false, return entries as an empty list."""

In [ ]:
from dotenv import load_dotenv

load_dotenv()

agent = Agent("openai-chat:gpt-4o-mini", output_type=SplitDecision, system_prompt=system_prompt)

result = await agent.run(
    "word: 'cuento' (Spanish), translations: ['Märchen', 'erzähle', 'zähle', 'Erzählung']"
)

print(result.output)